## Model & Tokenizer Specifications

### 🧠 Model Architecture (GPT-2)
The model follows a standardized transformer decoder-only architecture optimized for Sanskrit verse:
- **Type**: GPT-2 (Causal Language Model)
- **Layers**: 8 (Standardized for complexity tracking)
- **Attention Heads**: 8 (Capturing multi-scalar linguistic patterns)
- **Embedding Dimension**: 512
- **Context Window**: 512 tokens (Sufficient for full shloka context over multiple iterations)
- **Training Environment**: Trained on **Google Colab T4 GPU**
- **Total Parameters**: Approximately 50 Million

### 🔤 Tokenizer Details (Advanced Unigram)
To handle the unique sub-word nuances of Devanagari and Sanskrit sandhi rules, the following configuration is used:
- **Algorithm**: SentencePiece Unigram (Smoother sub-word extraction than BPE)
- **Vocabulary Size**: 32,000 tokens
- **Normalization**: NFKC (Standardizing Devanagari Unicode forms)
- **Pre-tokenization**: Metaspace (Replacement: `▁`) to preserve space-word relationships during synthesis.
- **Special Tokens**: `<s>` (BOS), `<pad>` (Padding), `</s>` (EOS), `<unk>` (Unknown), `<MBH>` (Mahabharata Style), `<RAM>` (Ramayana Style), `<eos>` (Verse End).

In [1]:
# 1. Environment Setup
%pip install -q -U transformers datasets sentencepiece accelerate evaluate tokenizers

In [2]:
import os
import torch
from google.colab import drive
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)
from datasets import load_dataset, concatenate_datasets
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import Metaspace

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
!nvidia-smi -L

Using device: cuda
GPU 0: Tesla T4 (UUID: GPU-dcbfb98d-03b9-73cf-5efb-e5d77a60e85f)


### Tokenizer Training (Advanced Unigram)

In [3]:
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Epic"
DATA_FILE = os.path.join(BASE_PATH, "data/processed/sanskrit_epic_dataset.txt")
TOKENIZER_FILE = os.path.join(BASE_PATH, "tokenizer/tokenizer.json")
os.makedirs(os.path.dirname(TOKENIZER_FILE), exist_ok=True)

tokenizer_obj = Tokenizer(models.Unigram())
tokenizer_obj.normalizer = normalizers.Sequence([NFKC()])
tokenizer_obj.pre_tokenizer = Metaspace(replacement="▁")

special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask" + ">", "<MBH>", "<RAM>", "<eos>"]
trainer = trainers.UnigramTrainer(
    vocab_size=32000,
    special_tokens=special_tokens,
    unk_token="<unk>"
)

print("Training SentencePiece Unigram Tokenizer...")
tokenizer_obj.train(files=[DATA_FILE], trainer=trainer)
tokenizer_obj.save(TOKENIZER_FILE)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_obj,
    bos_token="<s>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask" + ">"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training SentencePiece Unigram Tokenizer...


### Dataset Preparation & Augmentation

In [4]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)

# Dataset Augmentation: 2x Duplication to force pattern learning in small corpus
train_original = split_dataset["train"].shuffle(seed=42)
split_dataset["train"] = concatenate_datasets([train_original, train_original]).shuffle(seed=42)

print(f"Augmented Training set size: {len(split_dataset['train'])}")

Map:   0%|          | 0/98320 [00:00<?, ? examples/s]

Augmented Training set size: 176976


In [5]:
LOG_FILE = os.path.join(BASE_PATH, "training_log.txt")

class DetailedLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_world_process_zero and logs and "loss" in logs:
            kwargs['model'].eval()
            prompts = ["<MBH> ", "<RAM> "]
            samples = []
            for p in prompts:
                inputs = tokenizer(p, return_tensors="pt").to(device)
                with torch.no_grad():
                    outputs = kwargs['model'].generate(
                        inputs.input_ids, max_length=50, do_sample=True, temperature=0.8, top_p=0.95,
                        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id
                    )
                samples.append(tokenizer.decode(outputs[0], skip_special_tokens=False))
            kwargs['model'].train()

            log_entry = f"Step {state.global_step} | Loss: {logs['loss']:.4f}\nMBH: {samples[0]}\nRAM: {samples[1]}\n{'='*40}\n"
            print(log_entry)
            with open(LOG_FILE, "a", encoding="utf-8") as f: f.write(log_entry)

### Hyper-Precision Training Loop

In [6]:
config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=512,
    n_ctx=512,
    n_embd=512,
    n_layer=8,
    n_head=8,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)
model = GPT2LMHeadModel(config)

total_training_steps = (len(split_dataset["train"]) // 32) * 40
warmup_steps = int(total_training_steps * 0.1)

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_PATH, "model_output"),
    num_train_epochs=30,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    save_steps=10000,
    save_total_limit=1,
    eval_strategy="steps",
    eval_steps=10000,
    logging_steps=500,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=0.1,
    max_grad_norm=1.0,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    callbacks=[DetailedLogCallback(), EarlyStoppingCallback(early_stopping_patience=5)]
)

print(f"Starting Hyper-Precision training. Total steps est: {total_training_steps}")
trainer.train()

Starting Hyper-Precision training. Total steps est: 221200


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
10000,5.405338,5.356431
20000,4.842933,4.846762
30000,4.367174,4.465975
40000,4.015660,4.240154
50000,3.761206,4.106261
60000,3.558555,4.017337
70000,3.335251,3.993238
80000,3.135679,3.998986
90000,2.942618,4.037949
100000,2.784415,4.061265


Step 500 | Loss: 9.4112
MBH: <MBH> ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁दृश्यता
RAM: <RAM> ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁अतिथीनन्नपान ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁ ▁महाबुद्धे

Step 1000 | Loss: 8.1685
MBH: <MBH> ▁ । ▁ । ▁ । ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ । ▁ । ▁ <eos>

Step 1500 | Loss: 7.4216
MBH: <MBH> ▁ । ं ं ं ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ । ं ं ं ं ▁ । ं ▁ ॥ ▁ <eos>

Step 2000 | Loss: 6.9170
MBH: <MBH> ▁ ं ं ं ▁ । ं ▁च ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ । ं ं ं ं ं ं ः ▁ ॥ ▁ <eos>

Step 2500 | Loss: 6.5679
MBH: <MBH> ▁ ो ा ि ▁ म् ▁ । ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ ं ः ▁ ▁स ं ▁ । ं ं ं ं ं ं ▁ ॥ ▁ <eos>

Step 3000 | Loss: 6.3004
MBH: <MBH> ▁ ं ं ः ▁ । ं ि ः ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ ं ▁ । ं ▁ ं ः ः ▁ ॥ ▁ <eos>

Step 3500 | Loss: 6.0979
MBH: <MBH> ▁ ▁ े ▁ ▁च ि ▁ । म् ▁ ▁स ं म ं ▁च ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ ेन ं ▁च ▁ ▁ । ▁ ▁मे ो ▁स ▁ ▁ ▁न ो ं ि ▁ ॥ ▁ <eos>

Step 4000 | Loss: 5.9742
MBH: <MBH> ▁ । ▁तस्य ं ▁मे ▁च ▁न ं म्

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step 110500 | Loss: 2.7156
MBH: <MBH> ▁ ततोऽहं ▁पुनरेव ेदं ▁वचन ं ▁मम ▁पार्थिव ▁ । ▁मया ▁तु ▁मे ▁समनुज्ञात ो ▁मया ▁पुत्रेण ▁धीमता ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ ततोऽहं ▁राम मासाद्य ▁रामं ▁सत्यपराक्रम म् ▁ । ▁अभ्य नुज्ञाय ▁धर्मज्ञ ः ▁सह ▁स्त्रीभि ः ▁सह ▁संगत म् ▁ ॥ ▁ <eos>

Step 111000 | Loss: 2.6631
MBH: <MBH> ▁ यः ▁ शेषं ▁कुरुते ▁यत्न ं ▁कुरुते ▁लोक संक्षय म् ▁ । ▁तं ▁तु ▁संप्रेक्ष्य ▁तं ▁कालं ▁यो ▁न ▁जानात ि ▁भूमिप ः ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ जानामि ▁त्वा ▁वरारोहा ▁ यथेष्टं ▁दातुमर्हस ि ▁ । ▁न ▁त्वं ▁ विप्रियं ▁कृत्वा ▁तव ▁दुःख मनुत्तम म् ▁ ॥ ▁ <eos>

Step 111500 | Loss: 2.6440
MBH: <MBH> ▁ वनं ▁प्रविश्य ैव ▁तथा ▁ संवृते ; ▁तदा ▁नाशंस े ▁विजयाय ▁संजय ▁ । ▁यदाश्रौष ं ▁कर्ण मासाद्य ▁रथं ▁त ; ▁कर्णो ▁वैकर्तन ं ▁सह ेन्द्र सेन ▁ ॥ ▁ <eos>
RAM: <RAM> ▁ त्रिषु ▁लोक ेषु ▁विख्यात ो ▁य योर सुर ाधिप ▁ । ▁व्यक्त ं ▁लोक गतिं ▁प्राप्तो ▁मया ▁शत्रु ः ▁ त्वमागतः ▁ ॥ ▁ <eos>

Step 112000 | Loss: 2.6536
MBH: <MBH> ▁ तेषु ▁ तेषु ▁विचित्र ेषु ▁पर्वत ेषु ▁ गुहासु ▁च ▁ । ▁रावणः ▁सह ▁राजेन्द्र ो ▁गत ानाम कुतोभय ः

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=120000, training_loss=3.798125678507487, metrics={'train_runtime': 7881.3774, 'train_samples_per_second': 673.649, 'train_steps_per_second': 21.053, 'total_flos': 2.0231522273132544e+16, 'train_loss': 3.798125678507487, 'epoch': 21.695895859699874})

In [7]:
final_path = os.path.join(BASE_PATH, "sanskrit-gpt-epic-hyper")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Final Best Model saved to {final_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final Best Model saved to /content/drive/MyDrive/Epic/sanskrit-gpt-epic-hyper
